In [5]:
# Libraries
import pandas as pd                    # data handling (DataFrames)
import numpy as np                     # numerical operations
import matplotlib.pyplot as plt        # static charts (EDA)
import matplotlib.ticker as mtick      # axis number formats (£, %)
import seaborn as sns                  # used for the heatmap
from pathlib import Path               # file paths that work on Windows and Mac

import plotly.graph_objects as go      # interactive charts (dashboard)
from plotly.subplots import make_subplots
import ipywidgets as widgets           # dashboard filters
from IPython.display import display, HTML

In [7]:
# Display and chart style
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")   # thousands separator, 2 decimals
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlesize": 12})

# One consistent colour scheme for the whole notebook
BLUE, LIGHT_BLUE, ORANGE, GREY = "#2a78d6", "#9ec5f4", "#e8833a", "#9a9a9a"

# Axis formatter: 1440000 -> £1.4M, 25000 -> £25k
gbp = mtick.FuncFormatter(lambda x, _: f"£{x/1e6:.1f}M" if abs(x) >= 1e6 else
                          (f"£{x/1e3:.0f}k" if abs(x) >= 1e3 else f"£{x:.0f}"))

In [11]:
# The dataset is NOT included in this repository (excluded by .gitignore).
# Place Online_Retail.xlsx in the same folder as this notebook.
DATA_FILE = Path("Online_Retail.xlsx")

# InvoiceNo and StockCode are read as text because some values contain letters (C536379, 85123A)
raw = pd.read_excel(DATA_FILE, dtype={"InvoiceNo": str, "StockCode": str})
print(f"{raw.shape[0]:,} rows x {raw.shape[1]} columns")
raw.head()

541,909 rows x 8 columns


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,"17,850.00",United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,"17,850.00",United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom


## 2. Data quality assessment

Before changing anything I inspect the raw data to find problems. I check: **structure and data types**, **missing values**, **duplicates**, **cancelled transactions**, **negative or zero quantities and prices**, **rows that are not products**, **outliers** and **inconsistent labels**. Each check is quantified so the cleaning decisions in Section 3 are based on evidence.

In [14]:
# STRUCTURE CHECK
# First I look at every column: its data type and how many values are filled in.
# This tells me if pandas read the data correctly and where values are missing.
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [15]:
# FIRST LOOK AT THE NUMBERS
# describe() gives min, max, mean and quartiles. I use it to spot impossible values:
# a sale cannot have a negative quantity or a negative price.
raw[["Quantity", "UnitPrice"]].describe()

,Quantity,UnitPrice
count,"541,909.00","541,909.00"
mean,9.55,4.61
std,218.08,96.76
min,"-80,995.00","-11,062.06"
25%,1.00,1.25
50%,3.00,2.08
75%,10.00,4.13
max,"80,995.00","38,970.00"


The summary already shows problems: **negative quantities and prices**, and extreme values (±80,995 units, unit price of £38,970) that are far from the median (3 units at £2.08). These are investigated below.

In [16]:
# MISSING VALUES
# isna() marks empty cells as True. sum() counts them, mean() gives the proportion.
# I show the percentage because 1,000 missing values means little without knowing the total.
missing = pd.DataFrame({"missing": raw.isna().sum(),
                        "% of rows": raw.isna().mean() * 100})
missing[missing["missing"] > 0]

,missing,% of rows
Description,1454,0.27
CustomerID,135080,24.93


### 2.1 Investigating invalid and suspicious records

The summary statistics showed negative quantities, negative prices and extreme values. Below I investigate each problem step by step, to understand **what** these records are before deciding **how** to treat them.